# KPI 9 — Geographic Web Traffic

**requirements.md (Web & Traffic Analytics, KPI 9):** "Geographic Web Traffic: Web sessions by country/IP geolocation; identify top traffic sources."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_9_geographic_web_traffic` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per (**country_code**, **traffic_source**) — 12 countries (11 resolved ISO-2 codes + `unknown`) × 6 traffic sources where traffic exists, 52 rows. `country_code` from `dim_ip_geo` joined on `client_ip` (first-octet lookup; NULL `country_code` = unmapped IPv4 or non-IPv4 address, surfaced as `unknown` — never force-located, by design). `traffic_source` = HOST of the session's `entry_referer` (`parse_url(entry_referer, 'HOST')`, verbatim host incl. `www.`); NULL `entry_referer` = **direct** traffic, labeled `direct`. Note: `shop.b2bmarket.example` appears to be the platform's own domain (self-referral / internal navigation) — retained verbatim so its size stays visible.
* **Sessions:** from `fact_web_sessions` (user-confirmed rule: same `client_ip` + `user_agent`, split where the inactivity gap exceeds 30 minutes). **Bots are excluded** via `dim_device.is_bot` — view policy; the fact keeps bot traffic gross (28,038 bot sessions excluded here). Sessions with NULL user agent have no `dim_device` row and are retained as human.
* **Cell measures:** `session_count`; `source_share_within_country_pct` = 100 × cell sessions / country sessions (sums to ~100.00 per country); `event_count` = SUM(session `event_count`); `avg_events_per_session`; `avg_session_duration_seconds` = AVG(`duration_seconds`); `distinct_visitors` = COUNT(DISTINCT `client_ip`) per cell — **not additive across rows** (one visitor may appear in several countries and sources); `authenticated_session_count` = sessions with a `customer_sk`.
* **Country rollup (sessions by country):** `country_session_count` (repeats on every source row of the country; roll up with `GROUP BY country_code`); `country_session_share_pct` = 100 × country sessions / all human sessions (per-row ROUND — sums to ~100.00, not exactly, by rounding).
* **Traffic-source rollup (top traffic sources):** `traffic_source_session_count` (repeats on every country row of the source); `traffic_source_session_share_pct`; `traffic_source_rank` = ROW_NUMBER by source sessions DESC (tie-break `traffic_source` alphabetically — deterministic).
* **Verified totals:** 82,810 human sessions / 397,738 events (= 110,848 / 538,152 gross minus 28,038 bot sessions / 140,414 bot rows); every human session's `client_ip` matches `dim_ip_geo` (0 anti-join rows); 52 rows = 52 distinct (country, source) pairs.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_9_geographic_web_traffic
TBLPROPERTIES (
  'comment' = 'KPI 9 - Geographic web traffic (requirements.md: Web sessions by country/IP geolocation; identify top traffic sources). Grain: one row per (country_code, traffic_source) - country_code from dim_ip_geo joined on client_ip (first-octet lookup; NULL country_code = unmapped IPv4 or non-IPv4 address, surfaced as unknown - never force-located, by design); traffic_source = HOST of the session entry_referer via parse_url(entry_referer, HOST), verbatim host; NULL entry_referer = direct traffic, labeled direct; shop.b2bmarket.example is the platform own domain (self-referral), retained verbatim. Sessions from fact_web_sessions (user-confirmed rule: same client_ip + user_agent, split where the inactivity gap exceeds 30 minutes); bots EXCLUDED via dim_device.is_bot (view policy: the fact keeps bot traffic gross, 28,038 bot sessions excluded here); NULL-UA sessions retained as human. Cell measures: session_count; source_share_within_country_pct = 100 * cell sessions / country sessions (sums ~100.00 per country); event_count = SUM(session event_count); avg_events_per_session; avg_session_duration_seconds = AVG(duration_seconds); distinct_visitors = COUNT(DISTINCT client_ip) per cell (NOT additive across rows - one visitor may appear in several countries and sources); authenticated_session_count = sessions with a customer_sk. Country rollup: country_session_count (repeats on every source row of the country); country_session_share_pct = 100 * country sessions / all human sessions (per-row ROUND, sums ~100.00). Source rollup: traffic_source_session_count (repeats on every country row of the source); traffic_source_session_share_pct; traffic_source_rank = ROW_NUMBER by source sessions DESC (tie-break traffic_source alphabetically - deterministic). Verified: 52 rows = 12 countries (11 ISO-2 + unknown) x 6 sources; 82,810 human sessions / 397,738 events (110,848/538,152 gross minus 28,038 bot sessions/140,414 bot rows); 0 human sessions without a dim_ip_geo match.'
)
AS
WITH human_sessions AS (
  SELECT
    s.session_id,
    s.client_ip,
    s.customer_sk,
    s.event_count,
    s.duration_seconds,
    COALESCE(parse_url(s.entry_referer, 'HOST'), 'direct') AS traffic_source
  FROM b2b_ecommerce.gold.fact_web_sessions s
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON s.user_agent = dd.user_agent
  WHERE COALESCE(dd.is_bot, FALSE) = FALSE
),
geo_sessions AS (
  SELECT
    hs.*,
    COALESCE(g.country_code, 'unknown') AS country_code
  FROM human_sessions hs
  LEFT JOIN b2b_ecommerce.gold.dim_ip_geo g
    ON hs.client_ip = g.client_ip
),
cell AS (
  SELECT
    country_code,
    traffic_source,
    COUNT(*)                              AS session_count,
    SUM(event_count)                      AS event_count,
    ROUND(SUM(event_count) / COUNT(*), 2) AS avg_events_per_session,
    ROUND(AVG(duration_seconds), 1)       AS avg_session_duration_seconds,
    COUNT(DISTINCT client_ip)             AS distinct_visitors,
    SUM(CASE WHEN customer_sk IS NOT NULL THEN 1 ELSE 0 END)
                                           AS authenticated_session_count
  FROM geo_sessions
  GROUP BY country_code, traffic_source
),
source_rollup AS (
  SELECT
    traffic_source,
    SUM(session_count) AS traffic_source_session_count
  FROM cell
  GROUP BY traffic_source
),
source_ranked AS (
  SELECT
    traffic_source,
    traffic_source_session_count,
    ROUND(100 * traffic_source_session_count
              / SUM(traffic_source_session_count) OVER (), 2)
      AS traffic_source_session_share_pct,
    ROW_NUMBER() OVER (
      ORDER BY traffic_source_session_count DESC, traffic_source
    ) AS traffic_source_rank
  FROM source_rollup
)
SELECT
  c.country_code,
  c.traffic_source,
  c.session_count,
  ROUND(100 * c.session_count
            / SUM(c.session_count) OVER (PARTITION BY c.country_code), 2)
    AS source_share_within_country_pct,
  SUM(c.session_count) OVER (PARTITION BY c.country_code)
    AS country_session_count,
  ROUND(100 * SUM(c.session_count) OVER (PARTITION BY c.country_code)
            / SUM(c.session_count) OVER (), 2)
    AS country_session_share_pct,
  sr.traffic_source_session_count,
  sr.traffic_source_session_share_pct,
  sr.traffic_source_rank,
  c.event_count,
  c.avg_events_per_session,
  c.avg_session_duration_seconds,
  c.distinct_visitors,
  c.authenticated_session_count
FROM cell c
JOIN source_ranked sr
  ON sr.traffic_source = c.traffic_source

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 52 rows = 52 distinct
-- (country, source) pairs, 12 countries (11 resolved + unknown), 6 traffic
-- sources, SUM(session_count) = 82,810 (110,848 total sessions - 28,038 bot
-- sessions), SUM(event_count) = 397,738, per-country within-country share
-- sums = 100.00 +/- 0.03, country and source share sums ~100 (per-row ROUND,
-- tolerance 0.06), traffic_source_rank contiguous 1..6 and consistent with
-- source session counts, rollups recompute exactly from cell counts, 0 null
-- keys, 0 visitors > sessions, 0 authenticated > sessions.
WITH v AS (
  SELECT *,
         SUM(session_count) OVER (PARTITION BY country_code)   AS r_country_total,
         SUM(session_count) OVER (PARTITION BY traffic_source) AS r_source_total
  FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic
)
SELECT
  COUNT(*)                                                   AS rows_geo_source,
  COUNT(DISTINCT country_code || '|' || traffic_source)      AS distinct_pairs,
  COUNT(DISTINCT country_code)                               AS countries,
  COUNT(DISTINCT traffic_source)                             AS sources,
  SUM(session_count)                                         AS total_sessions,
  SUM(event_count)                                           AS total_events,
  MAX(traffic_source_rank)                                   AS max_source_rank,
  COUNT(DISTINCT traffic_source_rank)                        AS distinct_ranks,
  SUM(CASE WHEN country_code IS NULL OR traffic_source IS NULL
           THEN 1 ELSE 0 END)                                AS null_keys,
  SUM(CASE WHEN distinct_visitors > session_count
           THEN 1 ELSE 0 END)                               AS visitors_gt_sessions,
  SUM(CASE WHEN authenticated_session_count > session_count
           THEN 1 ELSE 0 END)                               AS auth_gt_sessions,
  SUM(CASE WHEN country_session_count <> r_country_total
           THEN 1 ELSE 0 END)                               AS country_rollup_mismatches,
  SUM(CASE WHEN traffic_source_session_count <> r_source_total
           THEN 1 ELSE 0 END)                               AS source_rollup_mismatches,
  (SELECT COUNT(*)
   FROM (
     SELECT traffic_source, traffic_source_rank, traffic_source_session_count,
            ROW_NUMBER() OVER (ORDER BY traffic_source_session_count DESC,
                               traffic_source) AS expected_rank
     FROM (SELECT DISTINCT traffic_source, traffic_source_rank,
                                  traffic_source_session_count
           FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic)
   ) WHERE traffic_source_rank <> expected_rank)      AS rank_mismatches,
  (SELECT ABS(ROUND(SUM(traffic_source_session_share_pct), 2) - 100)
   FROM (SELECT DISTINCT traffic_source, traffic_source_session_share_pct
         FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic))
                                                        AS source_share_sum_dev,
  (SELECT ABS(ROUND(SUM(country_share), 2) - 100)
   FROM (SELECT DISTINCT country_code, country_session_share_pct AS country_share
         FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic))
                                                        AS country_share_sum_dev,
  (SELECT MAX(dev)
   FROM (SELECT ABS(ROUND(SUM(source_share_within_country_pct), 2) - 100) AS dev
         FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic
         GROUP BY country_code))                     AS max_within_country_dev
FROM v

In [0]:
%sql
-- Verification: independently recompute every (country, source) cell through
-- a different path - sessions re-derived from RAW fact_web_logs events via
-- SQL window sessionization (same user-confirmed 30-min rule, no
-- fact_web_sessions), the entry referer taken as the first event's referer via
-- MIN(struct(event_time, log_id, referer)) instead of the fact's stored
-- entry_referer, the session's customer attributed only when exactly one
-- distinct customer appears, geo via dim_ip_geo, bots excluded via dim_device
-- - then FULL OUTER JOIN against the stored view and compare every cell.
-- Expected: 52 rows compared, 52/52 matches on sessions, events, visitors and
-- authenticated sessions, 0 rows only on either side.
WITH ev AS (
  SELECT log_id, client_ip, user_agent, event_time, referer, customer_sk
  FROM b2b_ecommerce.gold.fact_web_logs
),
lagged AS (
  SELECT e.*,
         LAG(e.event_time) OVER (
           PARTITION BY e.client_ip, e.user_agent
           ORDER BY e.event_time, e.log_id
         ) AS prev_time
  FROM ev e
),
marked AS (
  SELECT l.*,
         CASE WHEN prev_time IS NULL
               OR UNIX_TIMESTAMP(l.event_time) - UNIX_TIMESTAMP(l.prev_time) > 1800
              THEN 1 ELSE 0 END AS new_session
  FROM lagged l
),
sequenced AS (
  SELECT m.*,
         SUM(m.new_session) OVER (
           PARTITION BY m.client_ip, m.user_agent
           ORDER BY m.event_time, m.log_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
         ) AS session_seq
  FROM marked m
),
sessions2 AS (
  SELECT
    client_ip,
    user_agent,
    MIN(STRUCT(event_time, log_id, referer)) AS first_event,
    COUNT(*)                                 AS event_count,
    CASE WHEN COUNT(DISTINCT customer_sk) = 1
         THEN MAX(customer_sk) END            AS customer_sk
  FROM sequenced
  GROUP BY client_ip, user_agent, session_seq
),
human2 AS (
  SELECT s2.*
  FROM sessions2 s2
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON s2.user_agent = dd.user_agent
  WHERE COALESCE(dd.is_bot, FALSE) = FALSE
),
geo2 AS (
  SELECT
    COALESCE(g.country_code, 'unknown')                          AS country_code,
    COALESCE(parse_url(h2.first_event.referer, 'HOST'), 'direct') AS traffic_source,
    h2.event_count,
    h2.client_ip,
    h2.customer_sk
  FROM human2 h2
  LEFT JOIN b2b_ecommerce.gold.dim_ip_geo g
    ON h2.client_ip = g.client_ip
),
recompute AS (
  SELECT
    country_code,
    traffic_source,
    COUNT(*)                          AS r_session_count,
    SUM(event_count)                  AS r_event_count,
    COUNT(DISTINCT client_ip)         AS r_distinct_visitors,
    SUM(CASE WHEN customer_sk IS NOT NULL THEN 1 ELSE 0 END)
                                      AS r_authenticated_session_count
  FROM geo2
  GROUP BY country_code, traffic_source
),
cmp AS (
  SELECT
    COALESCE(v.country_code, rc.country_code)       AS country_code,
    COALESCE(v.traffic_source, rc.traffic_source)   AS traffic_source,
    v.session_count,
    rc.r_session_count,
    v.event_count,
    rc.r_event_count,
    v.distinct_visitors,
    rc.r_distinct_visitors,
    v.authenticated_session_count,
    rc.r_authenticated_session_count
  FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic v
  FULL OUTER JOIN recompute rc
    ON  rc.country_code   = v.country_code
   AND  rc.traffic_source = v.traffic_source
)
SELECT
  COUNT(*) AS rows_compared,
  SUM(CASE WHEN session_count IS NULL THEN 1 ELSE 0 END)
           AS view_only_rows,
  SUM(CASE WHEN r_session_count IS NULL THEN 1 ELSE 0 END)
           AS recompute_only_rows,
  SUM(CASE WHEN session_count <=> r_session_count THEN 1 ELSE 0 END)
           AS session_matches,
  SUM(CASE WHEN event_count <=> r_event_count THEN 1 ELSE 0 END)
           AS event_matches,
  SUM(CASE WHEN distinct_visitors <=> r_distinct_visitors THEN 1 ELSE 0 END)
           AS visitor_matches,
  SUM(CASE WHEN authenticated_session_count <=> r_authenticated_session_count
           THEN 1 ELSE 0 END)
           AS auth_matches,
  SUM(CASE WHEN session_count <=> r_session_count
            AND event_count <=> r_event_count
            AND distinct_visitors <=> r_distinct_visitors
            AND authenticated_session_count <=> r_authenticated_session_count
           THEN 1 ELSE 0 END)
           AS full_matches
FROM cmp

In [0]:
%sql
-- Headline: full geographic x traffic-source breakdown, top sources first
-- (the view is small - 52 rows).
SELECT *
FROM b2b_ecommerce.gold.kpi_9_geographic_web_traffic
ORDER BY traffic_source_rank, country_session_count DESC, country_code